# Study 806 — Prospect-Theory Value 🧠🎰

**Do stocks that look like an attractive *gamble* go on to earn *less*?**

Barberis, Mukherjee & Wang (2016) find that the **cumulative-prospect-theory (TK) value** of a
stock's recent return distribution predicts its cross-section of returns *negatively*: names
whose recent tape looks like a good gamble under Tversky-Kahneman — a right-skewed, lottery-like
distribution — carry a **high** TK value, are over-priced, and go on to under-earn. A long
**low-TK** / short **high-TK** book should earn a positive spread. We take the self-contained
daily version on a liquid US cross-section (2010-01-04 → 2026-06-30, 50 names,
trailing ≈5y TK value, monthly rebalance).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper bound.*


## 1. The idea in one picture

A prospect-theory investor evaluating a single stock as a standalone gamble uses a value function that is concave over gains, steeper over losses (loss aversion 2.25), and **probability weights** that overweight the tails. A right-skewed, lottery-like tape puts mass in the overweighted *upside* tail, so its **TK value is high** — the investor over-pays, and the future return is lower. Sort on the TK value; buy the boring low-TK names, sell the lottery tickets.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=138.7, t_nw=3.15, lo_bps=232.81, hi_bps=94.11, gross_sharpe=0.85)
print('long low-TK / short high-TK spread: %+.2f bps/month (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  low-TK book %+.2f bps vs high-TK book %+.2f bps'
      % (R['lo_bps'], R['hi_bps']))
print('  gross spread Sharpe (before cost, ann.): %.2f' % R['gross_sharpe'])

long low-TK / short high-TK spread: +138.70 bps/month (NW t = +3.15)
  low-TK book +232.81 bps vs high-TK book +94.11 bps
  gross spread Sharpe (before cost, ann.): 0.85


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`: a lottery-like tape both scores high TK *and* earns less) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, TK varies but is unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from prospect_theory import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=806, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0020, seed=806, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = -0.20  (should be ~0)
planted world: spread NW t = +2.82  (should light up)


## 3. The honest verdict — the famous edge *replicates* here

On this liquid mega-cap tape the long-low-TK / short-high-TK spread is **+138.70 bps/month** with NW *t* = **+3.15** — significant and with the **sign prospect theory predicts**: the boring low-TK names out-earned the lottery-like high-TK names over 2015–2026 (the permutation null centres at 0 with sd 33.9 bps; the observed value is ~4.1σ into the *right* tail). It holds in both eras (*t* = +2.00 / +2.48), and the seeded synthetic control recovers a planted relation cleanly — so this is a genuine replication. The spread even clears conservative costs (net **+124.53 bps/month** at 5 bps), but it leans on a survivorship-inflated short leg (the blown-up lottery names are absent) and a few hard-to-borrow lottery mega-caps. **Signal: Real**, **Tradability: Fragile.**